In [0]:
# Dependencias de esta etapa: Optuna para la búsqueda; LightGBM y XGBoost porque el módulo proyecto.models.candidates los importa.
# Va al principio: al instalar se reinicia el intérprete, así que hay que volver a ejecutar la celda siguiente en cada sesión.
%uv pip install optuna lightgbm xgboost

In [0]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar tu paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))


# 06 · Optimización con seguimiento (tracking)

**Proyecto RiskVision · Grupo 6.** Dataset: *Default of Credit Card Clients* (UCI, CC BY 4.0). Guía completa: `curso/kit-estudiante/guia-modelado-y-optimizacion.md`, secciones 3 y 4.

Buscar mejores hiperparámetros es probar muchas configuraciones. Sin registro, a la mitad ya no sabes
cuál era la buena: por eso cada ensayo queda en **MLflow**. Tres cuidados:

1. Se optimiza **solo el finalista** del torneo (notebook 05, sección 8): LightGBM con el paquete E. La regresión logística queda como contraste explicable y no se afina aquí.
2. El **espacio de búsqueda** lleva tu conocimiento del modelo, y el **presupuesto** de ensayos se fija antes.
3. La mejor puntuación de una búsqueda es **optimista**: se **comprueba** en entidades que la búsqueda no vio.

## 0. Cargar los datos de entrenamiento

In [0]:
import joblib
import numpy as np
import pandas as pd

from proyecto.config import ID_COL, PROCESSED_DIR, TARGET
from proyecto.data.split import assert_unique_ids, make_xy


def cargar_train(ruta):
    """Carga train.joblib. El archivo se creó con un pandas más viejo, cuyo pickle de Categorical
    pasa una tupla en vez de un dict: se traduce al formato actual solo durante la lectura."""
    try:
        return joblib.load(ruta)
    except NotImplementedError:
        _orig = pd.Categorical.__setstate__

        def _compat(self, state):
            if isinstance(state, tuple) and len(state) == 2:
                dtype, codes = state
                state = {"_ndarray": np.asarray(codes), "_dtype": dtype}
            return _orig(self, state)

        pd.Categorical.__setstate__ = _compat
        try:
            return joblib.load(ruta)
        finally:
            pd.Categorical.__setstate__ = _orig


# Las mismas constantes de todo el proyecto, en su único lugar (proyecto.config), como en los notebooks 03 y 05
train = cargar_train(PROCESSED_DIR / "train.joblib")       # lo generó el notebook 03; la prueba sigue cerrada
assert_unique_ids(train)                                   # cada cliente aparece una sola vez (reports/01_eda.md)
X, y = make_xy(train)
y = y.to_numpy()
listo = True

print(f"entrenamiento: {X.shape[0]:,} filas x {X.shape[1]} predictores (la prueba NO se abre hoy)")
print(f"objetivo: {TARGET!r} · tasa de incumplimiento: {y.mean():.2%}")

## 1. Pipeline y métrica (los del torneo)

Importa tu función de `src/proyecto/models/train.py` y define la **misma métrica principal** del torneo: la
optimización se mide con la métrica con la que elegiste a tus finalistas.

In [0]:
from lightgbm import LGBMClassifier

from proyecto.config import RANDOM_STATE
from proyecto.models.candidates import LGBM_PARAMS, make_lgbm       # parámetros de partida y fábrica del finalista
from proyecto.models.evaluate import repeated_folds, score_folds     # los mismos pliegues y la misma forma de medir del torneo
from proyecto.models.train import build_pipeline

# La métrica principal es la MISMA del torneo (notebook 05): ROC AUC sobre las probabilidades, mayor es mejor
METRICA = "roc_auc"
print("métrica principal:", METRICA)
print("parámetros de partida (candidates.py):", LGBM_PARAMS)

## 2. Conexión con el seguimiento

Usa el módulo de la plantilla tal como está diseñado: dentro de Databricks conecta al **tracking del workspace** (los experimentos se ven en la pestaña Experimentos); fuera de él (máquina local, como en la clase) guarda en `mlflow_local.db`.

In [0]:
try:
    from proyecto.tracking.mlflow_utils import configure_mlflow, log_run, set_experiment
except ModuleNotFoundError:
    configure_mlflow = None
    print("TODO: el módulo src/proyecto/tracking/mlflow_utils.py viene en la plantilla; si no está, vuelve a clonar la rama base.")

if configure_mlflow is None:
    donde = experimento = None
else:
    import os

    import mlflow

    en_databricks = bool(os.environ.get("DATABRICKS_HOST")) or Path("/Workspace").is_dir()
    if en_databricks:
        # Notebook dentro del workspace: tracking nativo de Databricks (experimentos visibles en la UI).
        # En serverless el módulo no detecta la conexión (DATABRICKS_RUNTIME_VERSION no existe) y su
        # sqlite local quedaría en /Repos, que es de solo lectura: se usa el tracking del workspace.
        mlflow.set_tracking_uri("databricks")
        mlflow.set_registry_uri("databricks-uc")
        donde = "databricks (dentro del workspace)"
    else:
        with warnings.catch_warnings(record=True) as avisos:   # en la máquina local: sqlite local del curso
            warnings.simplefilter("always")
            donde = configure_mlflow()
        for aviso in avisos:
            print("Aviso:", aviso.message)
    experimento = set_experiment("riskvision-optimizacion")   # el nombre del proyecto es RiskVision
    print("MLflow:", donde, "| experimento:", experimento)

> **Dónde queda el seguimiento:** fuera de Databricks, en `mlflow_local.db` (raíz del repo, ignorado por Git; se abre con
> `mlflow ui --backend-store-uri sqlite:///mlflow_local.db`). Dentro de Databricks, en el experimento del workspace
> (`riskvision-optimizacion`), visible en la pestaña Experimentos.

## 3. Reservar entidades que la búsqueda no verá

Separa, **dentro del entrenamiento**, un 25 % de las **entidades** (no de las filas). La búsqueda trabaja solo
con el resto; `test.joblib` sigue cerrado.

In [0]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split

if not listo:
    busqueda = reservado = None
    print("Sin datos: esta celda separará las entidades reservadas cuando estén.")
elif train[ID_COL].duplicated().any():
    # Por si los datos cambian: si un cliente apareciera en varias filas, se aparta por entidad, no por fila
    i_bus, i_res = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=7).split(train, groups=train[ID_COL]))
    busqueda, reservado = train.iloc[i_bus], train.iloc[i_res]
    print(f"búsqueda: {busqueda[ID_COL].nunique():,} clientes · reservados: {reservado[ID_COL].nunique():,} clientes")
else:
    # Un cliente por fila (comprobado al cargar con assert_unique_ids): reservar por fila es válido y
    # además permite estratificar por el objetivo, que solo incumple el 22 % (misma razón del notebook 03)
    busqueda, reservado = train_test_split(train, test_size=0.25, stratify=train[TARGET], random_state=7)
    print("Sin clientes repetidos: se reserva por fila, estratificado por el objetivo (válido según reports/01_eda.md).")

assert set(busqueda[ID_COL]).isdisjoint(set(reservado[ID_COL])), "hay clientes en los dos lados"
X_bus, y_bus = make_xy(busqueda)
X_res, y_res = make_xy(reservado)
y_bus, y_res = y_bus.to_numpy(), y_res.to_numpy()
print(f"búsqueda: {len(busqueda):,} clientes ({y_bus.mean():.2%} incumple) · "
      f"reservados: {len(reservado):,} clientes ({y_res.mean():.2%} incumple), que la búsqueda NO ve")

## 4. Espacio de búsqueda y presupuesto

Acota los rangos con lo que sabes del algoritmo (ver la tabla de la sección 3.2 de la guía) y fija el
**número de ensayos antes de empezar**.

In [0]:
from proyecto.features.build import FINAL_FEATURES   # el paquete E: las 5 variables elegidas en el notebook 04

N_ENSAYOS = 12          # presupuesto fijado ANTES de empezar: cada ensayo son 15 ajustes de LightGBM
FINALISTA = "LightGBM"  # el finalista del torneo con el paquete E (notebook 05, sección 8)

# Fijos (no se buscan): los mismos de candidates.py, para que partida y optimizado sean comparables
PARAMETROS_FIJOS = {"subsample_freq": 1, "random_state": RANDOM_STATE, "verbose": -1, "n_jobs": 4}


def espacio(trial):
    """Espacio de búsqueda de LightGBM, acotado con lo que se sabe del algoritmo (guía, §3.2):
    learning_rate bajo con más árboles, pocas hojas frente a ~18.000 filas de búsqueda,
    min_child_samples alto contra el sobreajuste de la clase rara (22 %) y regularización L1/L2."""
    return {
        "n_estimators": trial.suggest_int("n_estimators", 150, 600, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 7, 63),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 150),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        **PARAMETROS_FIJOS,
    }


print(f"Presupuesto: {N_ENSAYOS} ensayos · finalista: {FINALISTA} · variables: {len(FINAL_FEATURES)} nuevas del paquete E")

## 5. La búsqueda (un ensayo = una ejecución registrada)

Cada ensayo se registra con `log_run`: parámetros, métrica principal y etiquetas.

In [0]:
import optuna
from optuna.samplers import TPESampler

estudio = None
if not listo or busqueda is None:
    print("Sin datos: ejecuta antes las celdas de carga y de separación de entidades.")
else:
    # Los mismos 15 pliegues del torneo (5 estratificados x 3 repeticiones, semilla del proyecto),
    # pero calculados SOLO con los datos de búsqueda: las entidades reservadas no entran ni en un pliegue
    pliegues_busqueda = repeated_folds(X_bus, y_bus, n_splits=5, n_repeats=3, seed=RANDOM_STATE)

    def objetivo(trial):
        """Un ensayo = una configuración medida con los 15 pliegues y registrada como UNA ejecución en MLflow."""
        params = espacio(trial)
        por_pliegue = score_folds(lambda: build_pipeline(LGBMClassifier(**params), FINAL_FEATURES),
                                  X_bus, y_bus, pliegues_busqueda)
        metricas = {METRICA: por_pliegue[METRICA].mean(),
                    f"{METRICA}_std": por_pliegue[METRICA].std(),
                    "penalty": por_pliegue["penalty"].mean()}
        run_id = log_run(f"ensayo-{trial.number:02d}", params, metricas,
                         tags={"etapa": "06_optimizacion", "finalista": FINALISTA, "presupuesto": N_ENSAYOS})
        trial.set_user_attr("run_id", run_id)
        return metricas[METRICA]

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    estudio = optuna.create_study(direction="maximize", sampler=TPESampler(seed=RANDOM_STATE))
    estudio.optimize(objetivo, n_trials=N_ENSAYOS)          # el presupuesto se respeta: 12 ensayos, ni uno más

    print(f"mejor {METRICA} en la búsqueda (optimista, solo pliegues de búsqueda): {estudio.best_value:.4f}")
    print("mejores parámetros:", estudio.best_params)
    display(pd.Series(optuna.importance.get_param_importances(estudio)).round(3).sort_values(ascending=False))

## 6. La comprobación honesta

Entrena **dos** modelos con los datos de la búsqueda (parámetros de **partida** y **optimizados**) y compáralos
**en las entidades reservadas**, con un intervalo del 95 % que remuestrea **entidades completas**, no filas.

In [0]:
import numpy as np


def intervalo_por_entidad(diferencia_por_fila, entidades, n=1000, seed=42):
    """diferencia_por_fila = costo(optimizado) - costo(partida), fila a fila. Negativo: el optimizado mejora."""
    por_entidad = pd.DataFrame({"g": np.asarray(entidades), "d": np.asarray(diferencia_por_fila)}).groupby("g")["d"].agg(["sum", "count"])
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(por_entidad), size=(n, len(por_entidad)))
    medias = por_entidad["sum"].to_numpy()[idx].sum(axis=1) / por_entidad["count"].to_numpy()[idx].sum(axis=1)
    return np.percentile(medias, [2.5, 97.5])


from proyecto.models.evaluate import FN_WEIGHT, THEORETICAL_THRESHOLD, technical_metrics


def costo_por_fila(y_true, proba, threshold=THEORETICAL_THRESHOLD, fn_weight=FN_WEIGHT):
    """Costo 5:1 por cliente (0, 1 o 5): lo que aporta cada fila a la penalización relativa."""
    y_true, proba = np.asarray(y_true), np.asarray(proba, dtype=float)
    marcado = proba >= threshold
    return ((~marcado) & (y_true == 1)) * fn_weight + (marcado & (y_true == 0)).astype(int)


if estudio is None:
    bajo = alto = None
    print("Pendiente: lanza primero la búsqueda (sección 5).")
else:
    # Dos modelos ajustados SOLO con los datos de búsqueda: parámetros de partida vs. optimizados
    modelo_partida = make_lgbm(FINAL_FEATURES).fit(X_bus, y_bus)
    modelo_optimizado = build_pipeline(
        LGBMClassifier(**{**PARAMETROS_FIJOS, **estudio.best_params}), FINAL_FEATURES).fit(X_bus, y_bus)

    proba_partida = modelo_partida.predict_proba(X_res)[:, 1]
    proba_optimizado = modelo_optimizado.predict_proba(X_res)[:, 1]
    for nombre, proba in [("partida", proba_partida), ("optimizado", proba_optimizado)]:
        m = technical_metrics(y_res, proba)
        print(f"{nombre:11s} · {METRICA}: {m[METRICA]:.4f} · penalización 5:1: {m['penalty']:.4f} · "
              f"recall al revisar el 20 %: {m['recall_top']:.3f}")

    # Diferencia por cliente en las entidades reservadas: < 0 significa que el optimizado mejora
    diferencia_por_fila = costo_por_fila(y_res, proba_optimizado) - costo_por_fila(y_res, proba_partida)
    bajo, alto = intervalo_por_entidad(diferencia_por_fila, reservado[ID_COL].to_numpy())
    print(f"\nIntervalo del 95 % de la diferencia de costo 5:1 por cliente: [{bajo:+.4f}, {alto:+.4f}]")

**Regla de decisión escrita ANTES de mirar:** uso los parámetros optimizados solo si **todo el intervalo
del 95 % queda por debajo de cero** (diferencia de penalización 5:1 por cliente en las entidades reservadas);
si el intervalo cruza el cero, la mejora no se distingue del ruido y me quedo con los de partida.

**Resultado:** en los 6.000 clientes reservados, partida y optimizado empatan en AUC (0.7865) y la penalización 5:1 casi igual (0.549 vs. 0.543). El intervalo del 95 % de la diferencia de costo por cliente es [-0.0185, +0.0072]: cruza el cero, así que la mejora no se distingue del ruido y **me quedo con los parámetros de partida** de `candidates.py` (más simples y ya probados en el torneo). El umbral operativo se decide en la etapa 7.

## 7. Guardar la decisión

In [0]:
import json


def guardar_reporte(nombre, texto):
    """Guarda `nombre` en reports/ del repo. En la máquina local escribe directo; en Databricks, donde la
    carpeta del repo no es escribible por sistema de archivos, lo sube con la API del workspace."""
    carpeta = Path.cwd().resolve().parent / "reports"
    try:
        carpeta.mkdir(parents=True, exist_ok=True)
        (carpeta / nombre).write_text(texto, encoding="utf-8")
        return str(carpeta / nombre)
    except OSError:
        from databricks.sdk import WorkspaceClient
        from databricks.sdk.service.workspace import ImportFormat

        base = str(carpeta).removeprefix("/Workspace")      # /Repos/... ya es una ruta del workspace
        cliente = WorkspaceClient()
        cliente.workspace.mkdirs(base)
        cliente.workspace.upload(f"{base}/{nombre}", texto.encode("utf-8"), format=ImportFormat.AUTO, overwrite=True)
        return f"{base}/{nombre}"


if bajo is None:
    mejores_parametros = None
    print("Sin decisión todavía: esta celda guardará reports/06_best_params.json cuando la tomes.")
else:
    usar_optimizados = bool(alto < 0)   # la regla de la sección 6, escrita ANTES de mirar el intervalo
    origen = "optimizados" if usar_optimizados else "partida"
    mejores_parametros = {**PARAMETROS_FIJOS, **estudio.best_params} if usar_optimizados else dict(LGBM_PARAMS)
    registro = {"finalista": FINALISTA, "origen": origen, "parametros": mejores_parametros,
                "tracking": donde, "experimento": experimento, "n_ensayos": N_ENSAYOS,
                "intervalo_95": [float(bajo), float(alto)]}
    ruta = guardar_reporte("06_best_params.json", json.dumps(registro, indent=2, default=float))
    print("guardado:", ruta, "· parámetros de", origen)

In [0]:
from pathlib import Path

informe = Path.cwd().resolve().parent / "reports" / "06_optimizacion.md"
if bajo is None or mejores_parametros is None:
    print("Sin resultados: se escribirá reports/06_optimizacion.md al ejecutar el notebook completo.")
else:
    importancias = optuna.importance.get_param_importances(estudio)
    texto = [
        "# 06 · Optimización con seguimiento",
        "",
        f"- **Finalista:** {FINALISTA} con el paquete E (elegido en el notebook 05).",
        f"- **Métrica principal:** {METRICA} (la misma del torneo). **Presupuesto:** {N_ENSAYOS} ensayos, respetado.",
        f"- **Mejor {METRICA} en la búsqueda** (optimista, solo pliegues de búsqueda): {estudio.best_value:.4f}.",
        f"- **Mejores parámetros:** {estudio.best_params}.",
        "- **Importancia de los hiperparámetros:** " + ", ".join(f"{k} {v:.2f}" for k, v in importancias.items()),
        f"- **Comprobación honesta** en {len(reservado):,} clientes reservados que la búsqueda no vio: "
        f"intervalo del 95 % de la diferencia de costo 5:1 por cliente [{bajo:+.4f}, {alto:+.4f}].",
        f"- **Decisión** (regla escrita antes de mirar): parámetros de **{origen}**, porque "
        + ("todo el intervalo quedó por debajo de cero." if alto < 0 else "el intervalo cruza el cero: la mejora no se distingue del ruido."),
        f"- **Seguimiento:** cada ensayo es una ejecución registrada ({experimento}, {donde}); umbral operativo pendiente para la etapa 7.",
    ]
    ruta = guardar_reporte("06_optimizacion.md", "\n".join(texto) + "\n")
    print("guardado:", ruta)

## 8. Comprobar el seguimiento

Abre tu experimento con `mlflow ui --backend-store-uri
sqlite:///mlflow_local.db`: ¿están **todos** los ensayos? ¿cada uno con sus parámetros y métricas? ¿puedes
ordenarlos por la métrica principal?

## Entregable de esta etapa

- [ ] **Un ensayo = una ejecución registrada**, con parámetros, métrica y etiquetas.
- [ ] El **presupuesto** de ensayos está escrito y se respetó.
- [ ] Hay **entidades reservadas** que la búsqueda nunca vio.
- [ ] La **comprobación** (intervalo del 95 % remuestreando entidades) está hecha y la regla de decisión
  estaba escrita antes.
- [ ] `reports/06_optimizacion.md` y `reports/06_best_params.json` completados.